# Lab 2 — Partea C: Optimizarea hiperparametrilor și Bonus

Această secțiune se concentrează pe îmbunătățirea performanței modelului prin ajustarea hiperparametrilor. Vom folosi strategia optimă de pregătire a datelor stabilită în Partea B (codificare ordinală, cele 3 *feature*-uri inginerite și ponderi balansate pentru clase).

Vom parcurge următorii pași:
1. **Tuning manual:** Ajustarea intuitivă a hiperparametrilor cheie (`max_depth`, `learning_rate`, `n_estimators`) și analizarea compromisului *bias-variance*.
2. **Optimizare Bayesiană cu Optuna (`TPESampler`):** Căutarea automată a celei mai bune combinații de hiperparametri, generarea graficelor de optimizare și salvarea pașilor intermediari.
3. **Bonus:** Implementarea unui al doilea algoritm de tip *boosted trees* (LightGBM) pe exact aceleași date și compararea rezultatelor.


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
import optuna
from sklearn.model_selection import StratifiedKFold

# Importăm funcțiile gata pregătite din lab2_utils.py
from lab2_utils import get_final_data, get_split, balanced_weights, cv_macro_f1

# 1. Încărcăm datele finale (conțin deja cele 3 feature-uri noi și codificarea ordinală)
X, y = get_final_data()

# 2. Facem împărțirea fixă train/test (80/20, stratificată)
X_train, X_test, y_train, y_test = get_split(X, y)

# 3. Calculăm ponderile balansate pentru clase
sample_weights_train = balanced_weights(y_train)

print("Date încărcate cu succes prin lab2_utils!")
print("Forma matricei de antrenare (X_train):", X_train.shape)
print("Distribuția claselor în y_train:\n", y_train.value_counts().sort_index())

Date încărcate cu succes prin lab2_utils!
Forma matricei de antrenare (X_train): (2344, 253)
Distribuția claselor în y_train:
 PriceClass
0    189
1    407
2    830
3    549
4    318
5     51
Name: count, dtype: int64


## 2. Optimizare Bayesiană cu Optuna (`TPESampler`)

Pentru a găsi combinația ideală de hiperparametri fără o căutare oarbă (*grid search*), folosim librăria `optuna` împreună cu **`TPESampler`** (Tree-structured Parzen Estimator), algoritm care corespunde optimizării bayesiene.

Optimizăm următorii parametri:
- `max_depth`: între 3 și 9
- `learning_rate`: între 0.01 și 0.2 (scară logaritmică)
- `n_estimators`: între 50 și 300
- `subsample` și `colsample_bytree`: între 0.6 și 1.0
- `min_child_weight`: între 1 și 5
- `gamma`: între 0.0 și 1.0

In [5]:
import optuna
from sklearn.model_selection import StratifiedKFold

# Ascundem mesajele repetitive de logare Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 5),
        "gamma": trial.suggest_float("gamma", 0.0, 1.0),
        "random_state": 42,
        "n_jobs": -1
    }
    
    model = XGBClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Folosim funcția cv_macro_f1 din lab2_utils care aplică automat și sample weights
    scores = cv_macro_f1(model, X_train, y_train, cv=cv, weights=True)
    return scores.mean()

# Inițializăm TPESampler conform cerinței din laborator[cite: 11]
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)

# Rulăm optimizarea (30 de trial-uri)
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Cel mai bun scor Macro-F1 găsit:", round(study.best_value, 4))
print("\nCei mai buni hiperparametri găsiți:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

  0%|          | 0/30 [00:00<?, ?it/s]

Cel mai bun scor Macro-F1 găsit: 0.7665

Cei mai buni hiperparametri găsiți:
  max_depth: 6
  learning_rate: 0.09665369441538829
  n_estimators: 225
  subsample: 0.6799648837552809
  colsample_bytree: 0.9129773409175954
  min_child_weight: 4
  gamma: 0.0034898483696720883


### Vizualizarea procesului de optimizare și salvarea studiului

Generăm graficele pentru istoricul optimizării și importanța hiperparametrilor, salvând totodată studiul Optuna pe disc.

In [6]:
import joblib
from optuna.visualization import plot_optimization_history, plot_param_importances

# 1. Generăm graficele interactive Plotly pentru Optuna
fig_hist = plot_optimization_history(study)
fig_hist.update_layout(title="Istoricul optimizării Optuna (Macro-F1)")
fig_hist.show()

fig_imp = plot_param_importances(study)
fig_imp.update_layout(title="Importanța hiperparametrilor în optimizare")
fig_imp.show()

# 2. Salvăm studiul pe disc (pașii intermediari)
joblib.dump(study, "optuna_ames_study.pkl")
print("Studiul Optuna a fost salvat cu succes în fișierul 'optuna_ames_study.pkl'.")

Studiul Optuna a fost salvat cu succes în fișierul 'optuna_ames_study.pkl'.


## 3. Tuning manual de hiperparametri și compromisul Bias-Variance

Înainte de optimizarea automată, testăm manual câțiva hiperparametri pentru a observa cum se modifică rezultatele și cum influențează aceștia compromisul *bias-variance*:
- **`max_depth`**: O adâncime prea mică duce la *underfitting* (bias mare), iar una prea mare duce la *overfitting* (varianță mare).
- **`learning_rate`**: Rata de învățare a modelului.

In [7]:
from sklearn.model_selection import StratifiedKFold

# Testăm manual câteva seturi de hiperparametri de bază
manual_configs = {
    "Baseline (default)": XGBClassifier(random_state=42),
    "Adâncime mică (max_depth=3)": XGBClassifier(max_depth=3, random_state=42),
    "Adâncime mare (max_depth=8, lr=0.2)": XGBClassifier(max_depth=8, learning_rate=0.2, random_state=42),
    "Rată învățare mică (lr=0.05, n=200)": XGBClassifier(learning_rate=0.05, n_estimators=200, random_state=42),
}

manual_results = []
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in manual_configs.items():
    scores = cv_macro_f1(model, X_train, y_train, cv=cv5, weights=True)
    manual_results.append({"Configurație": name, "Mean Macro-F1": scores.mean(), "Std": scores.std()})

df_manual = pd.DataFrame(manual_results).sort_values("Mean Macro-F1", ascending=False).round(4)
display(df_manual)

,Configurație,Mean Macro-F1,Std
2,"Adâncime mare (max_depth=8, lr=0.2)",0.7535,0.0161
1,Adâncime mică (max_depth=3),0.7467,0.0204
0,Baseline (default),0.7462,0.0185
3,"Rată învățare mică (lr=0.05, n=200)",0.7293,0.0191


## 4. BONUS: Implementarea și optimizarea LightGBM

Implementăm un al doilea algoritm bazat pe arbori (*boosted trees*) — **LightGBM** (`LGBMClassifier`) — pe exact aceleași date și caracteristici, comparând rezultatele finale cu XGBoost.

In [11]:
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

# Antrenăm LightGBM folosind aceleași date și ponderi balansate
lgb_model = LGBMClassifier(random_state=42, verbose=-1)
lgb_model.fit(X_train, y_train, sample_weight=sample_weights_train)

# Evaluăm prin 5-fold CV
lgb_scores = cv_macro_f1(lgb_model, X_train, y_train, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), weights=True)
print(f"LightGBM 5-fold CV Macro-F1: {lgb_scores.mean():.4f} +/- {lgb_scores.std():.4f}")

# Comparație directă pe setul de test între XGBoost optimizat (prin Optuna) și LightGBM
best_xgb = XGBClassifier(**study.best_params, random_state=42)
best_xgb.fit(X_train, y_train, sample_weight=sample_weights_train)

xgb_test_pred = best_xgb.predict(X_test)
lgb_test_pred = lgb_model.predict(X_test)

comparison_df = pd.DataFrame({
    "Model": ["XGBoost (Optimizat Optuna)", "LightGBM (Baseline)"],
    "Test Macro-F1": [
        f1_score(y_test, xgb_test_pred, average="macro"),
        f1_score(y_test, lgb_test_pred, average="macro")
    ]
})
comparison_df.round(4)

LightGBM 5-fold CV Macro-F1: 0.7492 +/- 0.0166


,Model,Test Macro-F1
0,XGBoost (Optimizat Optuna),0.7460
1,LightGBM (Baseline),0.7398
